In [ ]:
import pyspark
pyspark.__version__

'4.0.2'

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.appName("SouthWest_MRF").getOrCreate()

In [ ]:
from pyspark.sql.functions import col

In [ ]:
from pyspark.sql import functions as F

In [ ]:
df = spark.read.csv("/content/233059262_southwest-healthcare-system_standardcharges.csv",header=False, inferSchema=False)

In [ ]:
df.show(10)

+--------------------+---------------+-----------+--------------------+--------------------+-----------------+--------------------+--------------------+---------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------

In [ ]:
df.show(5, truncate=False)

+------------------------------+---------------+-----------+-----------------------------------------------------------------+-------------------------------------------------------------------------------------------------+-----------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------+---------+-----------------------+---------------------------------+---------------------+---------------------+------------------------------------------------------+------------------------------------------------+----------------------------------------------------------+---------------------------------------------------------+-------------------------------------+--------------------------------------------+--------------------------------------------------+----------------

In [ ]:
# extracted the header
header_row = df.take(3)[2]

header = header_row.asDict().values()
header = [str(col).strip() for col in header]

print("Header preview:")
print(header[:20])

Header preview:
['description', 'code|1', 'code|1|type', 'setting', 'code|2', 'code|2|type', 'drug_unit_of_measurement', 'drug_type_of_measurement', 'modifiers', 'standard_charge|gross', 'standard_charge|discounted_cash', 'standard_charge|min', 'standard_charge|max', 'standard_charge|AETNA | MANAGED CARE|negotiated_dollar', 'standard_charge|AETNA | MANAGED CARE|methodology', 'standard_charge|AETNA | MANAGED CARE|negotiated_percentage', 'standard_charge|AETNA | MANAGED CARE|negotiated_algorithm', 'estimated_amount|AETNA | MANAGED CARE', 'additional_payer_notes |AETNA | MANAGED CARE', 'standard_charge|AETNA | MEDICARE|negotiated_dollar']


In [ ]:
from pyspark.sql.functions import monotonically_increasing_id

In [ ]:
# removing the first 3 rows
df_ = df.rdd.zipWithIndex() \
    .filter(lambda row: row[1] > 2) \
    .map(lambda row: row[0]) \
    .toDF(schema=df.schema)

df_.show(5)

+--------------------+-----+-----+----+--------+---+----+----+-------+----+-----+---------+------+-------+--------------------+----+----+----+----+----+----+----+----+----+----+-------+--------------------+----+----+----+----+-------+------------+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-------+--------------------+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-------+--------------------+----+----+----+----+---------+------------+----+-----+-----+-----+-------+--------------------+-----+-----+-----+-----+--------+--------------------+-----+-----+-----+-----+--------+------------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-------+------------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-------+--------------------+-----+-----+--

In [ ]:
# applied the header names
df_ = df_.toDF(*header)

df_.show(5)

+--------------------+------+-----------+-------+--------+-----------+------------------------+------------------------+---------+---------------------+-------------------------------+-------------------+-------------------+------------------------------------------------------+------------------------------------------------+----------------------------------------------------------+---------------------------------------------------------+-------------------------------------+--------------------------------------------+--------------------------------------------------+--------------------------------------------+------------------------------------------------------+-----------------------------------------------------+---------------------------------+----------------------------------------+------------------------------------------------------------------------------+------------------------------------------------------------------------+----------------------------------------

In [ ]:
#  checking the column names
print(df_.columns[:20])

['description', 'code|1', 'code|1|type', 'setting', 'code|2', 'code|2|type', 'drug_unit_of_measurement', 'drug_type_of_measurement', 'modifiers', 'standard_charge|gross', 'standard_charge|discounted_cash', 'standard_charge|min', 'standard_charge|max', 'standard_charge|AETNA | MANAGED CARE|negotiated_dollar', 'standard_charge|AETNA | MANAGED CARE|methodology', 'standard_charge|AETNA | MANAGED CARE|negotiated_percentage', 'standard_charge|AETNA | MANAGED CARE|negotiated_algorithm', 'estimated_amount|AETNA | MANAGED CARE', 'additional_payer_notes |AETNA | MANAGED CARE', 'standard_charge|AETNA | MEDICARE|negotiated_dollar']


In [ ]:
df_.printSchema()

root
 |-- description: string (nullable = true)
 |-- code|1: string (nullable = true)
 |-- code|1|type: string (nullable = true)
 |-- setting: string (nullable = true)
 |-- code|2: string (nullable = true)
 |-- code|2|type: string (nullable = true)
 |-- drug_unit_of_measurement: string (nullable = true)
 |-- drug_type_of_measurement: string (nullable = true)
 |-- modifiers: string (nullable = true)
 |-- standard_charge|gross: string (nullable = true)
 |-- standard_charge|discounted_cash: string (nullable = true)
 |-- standard_charge|min: string (nullable = true)
 |-- standard_charge|max: string (nullable = true)
 |-- standard_charge|AETNA | MANAGED CARE|negotiated_dollar: string (nullable = true)
 |-- standard_charge|AETNA | MANAGED CARE|methodology: string (nullable = true)
 |-- standard_charge|AETNA | MANAGED CARE|negotiated_percentage: string (nullable = true)
 |-- standard_charge|AETNA | MANAGED CARE|negotiated_algorithm: string (nullable = true)
 |-- estimated_amount|AETNA | MANAG

In [ ]:
df_ = df_.toDF(*header)
df_.show(5, truncate=False)
print(df_.columns[:20])

+------------------------------+------+-----------+-------+--------+-----------+------------------------+------------------------+---------+---------------------+-------------------------------+-------------------+-------------------+------------------------------------------------------+------------------------------------------------+----------------------------------------------------------+---------------------------------------------------------+-------------------------------------+--------------------------------------------+--------------------------------------------------+--------------------------------------------+------------------------------------------------------+-----------------------------------------------------+---------------------------------+----------------------------------------+------------------------------------------------------------------------------+------------------------------------------------------------------------+------------------------------

In [ ]:
#  cleaning the column names
clean_cols = [c.strip().lower().replace(" ", "_").replace("-", "_") for c in df_.columns]
df_ = df_.toDF(*clean_cols)

print(df_.columns[:20])

['description', 'code|1', 'code|1|type', 'setting', 'code|2', 'code|2|type', 'drug_unit_of_measurement', 'drug_type_of_measurement', 'modifiers', 'standard_charge|gross', 'standard_charge|discounted_cash', 'standard_charge|min', 'standard_charge|max', 'standard_charge|aetna_|_managed_care|negotiated_dollar', 'standard_charge|aetna_|_managed_care|methodology', 'standard_charge|aetna_|_managed_care|negotiated_percentage', 'standard_charge|aetna_|_managed_care|negotiated_algorithm', 'estimated_amount|aetna_|_managed_care', 'additional_payer_notes_|aetna_|_managed_care', 'standard_charge|aetna_|_medicare|negotiated_dollar']


In [ ]:
df_.printSchema()

root
 |-- description: string (nullable = true)
 |-- code|1: string (nullable = true)
 |-- code|1|type: string (nullable = true)
 |-- setting: string (nullable = true)
 |-- code|2: string (nullable = true)
 |-- code|2|type: string (nullable = true)
 |-- drug_unit_of_measurement: string (nullable = true)
 |-- drug_type_of_measurement: string (nullable = true)
 |-- modifiers: string (nullable = true)
 |-- standard_charge|gross: string (nullable = true)
 |-- standard_charge|discounted_cash: string (nullable = true)
 |-- standard_charge|min: string (nullable = true)
 |-- standard_charge|max: string (nullable = true)
 |-- standard_charge|aetna_|_managed_care|negotiated_dollar: string (nullable = true)
 |-- standard_charge|aetna_|_managed_care|methodology: string (nullable = true)
 |-- standard_charge|aetna_|_managed_care|negotiated_percentage: string (nullable = true)
 |-- standard_charge|aetna_|_managed_care|negotiated_algorithm: string (nullable = true)
 |-- estimated_amount|aetna_|_manag

In [ ]:
# checking missing values

df_.select([
    F.count(F.when(col(c).isNull(), c)).alias(c) for c in df_.columns
]).show()

+-----------+------+-----------+-------+------+-----------+------------------------+------------------------+---------+---------------------+-------------------------------+-------------------+-------------------+------------------------------------------------------+------------------------------------------------+----------------------------------------------------------+---------------------------------------------------------+-------------------------------------+--------------------------------------------+--------------------------------------------------+--------------------------------------------+------------------------------------------------------+-----------------------------------------------------+---------------------------------+----------------------------------------+------------------------------------------------------------------------------+------------------------------------------------------------------------+---------------------------------------------------

In [ ]:
df_.dropna(how="all").show(5)

+--------------------+------+-----------+-------+--------+-----------+------------------------+------------------------+---------+---------------------+-------------------------------+-------------------+-------------------+------------------------------------------------------+------------------------------------------------+----------------------------------------------------------+---------------------------------------------------------+-------------------------------------+--------------------------------------------+--------------------------------------------------+--------------------------------------------+------------------------------------------------------+-----------------------------------------------------+---------------------------------+----------------------------------------+------------------------------------------------------------------------------+------------------------------------------------------------------------+----------------------------------------

In [ ]:
# see the cleaned column names first
print(df_.columns[:30])

['description', 'code|1', 'code|1|type', 'setting', 'code|2', 'code|2|type', 'drug_unit_of_measurement', 'drug_type_of_measurement', 'modifiers', 'standard_charge|gross', 'standard_charge|discounted_cash', 'standard_charge|min', 'standard_charge|max', 'standard_charge|aetna_|_managed_care|negotiated_dollar', 'standard_charge|aetna_|_managed_care|methodology', 'standard_charge|aetna_|_managed_care|negotiated_percentage', 'standard_charge|aetna_|_managed_care|negotiated_algorithm', 'estimated_amount|aetna_|_managed_care', 'additional_payer_notes_|aetna_|_managed_care', 'standard_charge|aetna_|_medicare|negotiated_dollar', 'standard_charge|aetna_|_medicare|methodology', 'standard_charge|aetna_|_medicare|negotiated_percentage', 'standard_charge|aetna_|_medicare|negotiated_algorithm', 'estimated_amount|aetna_|_medicare', 'additional_payer_notes_|aetna_|_medicare', 'standard_charge|anthem_blue_cross_blue_shield_|_managed_care|negotiated_dollar', 'standard_charge|anthem_blue_cross_blue_shield

In [ ]:
clean_cols = [
    c.replace("|", "_").replace(" ", "_").lower()
    for c in df_.columns
]

df_ = df_.toDF(*clean_cols)

print(df_.columns[:20])
print("Total columns in df_:", len(df_.columns))

['description', 'code_1', 'code_1_type', 'setting', 'code_2', 'code_2_type', 'drug_unit_of_measurement', 'drug_type_of_measurement', 'modifiers', 'standard_charge_gross', 'standard_charge_discounted_cash', 'standard_charge_min', 'standard_charge_max', 'standard_charge_aetna___managed_care_negotiated_dollar', 'standard_charge_aetna___managed_care_methodology', 'standard_charge_aetna___managed_care_negotiated_percentage', 'standard_charge_aetna___managed_care_negotiated_algorithm', 'estimated_amount_aetna___managed_care', 'additional_payer_notes__aetna___managed_care', 'standard_charge_aetna___medicare_negotiated_dollar']
Total columns in df_: 236


In [ ]:
price_cols = [
    "standard_charge_gross",
    "standard_charge_discounted_cash",
    "standard_charge_min",
    "standard_charge_max"
]

for c in price_cols:
    df_ = df_.withColumn(c, F.regexp_replace(col(c), ",", ""))
    df_ = df_.withColumn(c, col(c).cast("double"))

In [ ]:
df_.select(price_cols).show(5, truncate=False)
print("Total columns in df_ after cleaning:", len(df_.columns))

+---------------------+-------------------------------+-------------------+-------------------+
|standard_charge_gross|standard_charge_discounted_cash|standard_charge_min|standard_charge_max|
+---------------------+-------------------------------+-------------------+-------------------+
|182.0                |72.8                           |14.5554            |127.4              |
|1684.0               |673.6                          |117.2184           |1397.3             |
|488.0                |195.2                          |141.79576          |341.6              |
|391.0                |156.4                          |116.127            |273.7              |
|293.0                |117.2                          |87.021             |205.1              |
+---------------------+-------------------------------+-------------------+-------------------+
only showing top 5 rows
Total columns in df_ after cleaning: 236


# Creating df_main for smaller calculation by filtering out the columns which are only necessary

In [ ]:
df_main = df_.select(
    "description",
    "code_1",
    "code_1_type",
    "setting",
    "standard_charge_gross",
    "standard_charge_discounted_cash",
    "standard_charge_min",
    "standard_charge_max"
)

In [ ]:
df_main.printSchema()
df_main.show(5, truncate=False)

root
 |-- description: string (nullable = true)
 |-- code_1: string (nullable = true)
 |-- code_1_type: string (nullable = true)
 |-- setting: string (nullable = true)
 |-- standard_charge_gross: double (nullable = true)
 |-- standard_charge_discounted_cash: double (nullable = true)
 |-- standard_charge_min: double (nullable = true)
 |-- standard_charge_max: double (nullable = true)

+------------------------------+------+-----------+-------+---------------------+-------------------------------+-------------------+-------------------+
|description                   |code_1|code_1_type|setting|standard_charge_gross|standard_charge_discounted_cash|standard_charge_min|standard_charge_max|
+------------------------------+------+-----------+-------+---------------------+-------------------------------+-------------------+-------------------+
|OT ICE MASSAGE EA 15 MIN      |97039 |HCPCS      |Both   |182.0                |72.8                           |14.5554            |127.4             

Drop rows when gross priuce is missing

In [ ]:
df_main = df_main.dropna(subset=["standard_charge_gross"])

created price range

In [ ]:
df_main = df_main.withColumn(
    "price_range",
    col("standard_charge_max") - col("standard_charge_min")
)
df_main.select("description", "price_range").show(5, truncate=False)

+------------------------------+------------------+
|description                   |price_range       |
+------------------------------+------------------+
|OT ICE MASSAGE EA 15 MIN      |112.8446          |
|SP FIBEROPT SWAL EVAL CINE/VID|1280.0816         |
|PT EVAL HIGH COMPLEXITY       |199.80424000000002|
|PT EVAL MODERATE COMPLEXITY   |157.57299999999998|
|PT EVAL LOW COMPLEXITY        |118.079           |
+------------------------------+------------------+
only showing top 5 rows


**Saving full cleaned datasets**

In [ ]:
df_.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("clean_hospital_full_data")

**Saving Smaller Analysis datasets**

In [ ]:
df_main.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("clean_hospital_pricing")

In [ ]:
import os

print("Full data folder:", os.listdir("clean_hospital_full_data"))
print("Filtered data folder:", os.listdir("clean_hospital_pricing"))

Full data folder: ['part-00000-11d3f49c-3a6b-42a5-b816-990f98bedb9b-c000.csv', '.part-00000-11d3f49c-3a6b-42a5-b816-990f98bedb9b-c000.csv.crc', '._SUCCESS.crc', '_SUCCESS']
Filtered data folder: ['._SUCCESS.crc', '.part-00000-12dc8ccf-36d5-4c80-a81f-471acb318d51-c000.csv.crc', 'part-00000-12dc8ccf-36d5-4c80-a81f-471acb318d51-c000.csv', '_SUCCESS']


In [ ]:
import os
import shutil

full_folder = "clean_hospital_full_data"
full_csv = [f for f in os.listdir(full_folder) if f.startswith("part-") and f.endswith(".csv")][0]
shutil.copy(os.path.join(full_folder, full_csv), "clean_hospital_full_data.csv")

main_folder = "clean_hospital_pricing"
main_csv = [f for f in os.listdir(main_folder) if f.startswith("part-") and f.endswith(".csv")][0]
shutil.copy(os.path.join(main_folder, main_csv), "clean_hospital_pricing.csv")

'clean_hospital_pricing.csv'

In [ ]:
from google.colab import files

files.download("clean_hospital_full_data.csv")
files.download("clean_hospital_pricing.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>